# 17 · Run records of the jittered-crop runs (CPU, about 2 min)
Runtime: CPU. Reads summary.json and history.csv of all 190 runs (read only) and writes one small table `results/p2_jit_eval/run_records.csv`. Nothing else is changed.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, glob, json, numpy as np, pandas as pd
PROJECT = '/content/drive/MyDrive/quantum-placement'
ROOTS = {'train': f'{PROJECT}/results/p2_jit_runs_train', 'val': f'{PROJECT}/results/p2_jit_runs_val'}
rows = []
for sched, root in ROOTS.items():
    for d in sorted(glob.glob(f'{root}/*_fold*_seed*')):
        p = f'{d}/summary.json'
        if not os.path.exists(p): continue
        s = json.load(open(p)); name = os.path.basename(d)
        r = dict(schedule=sched, run=name, model=s.get('model'), fold=s.get('fold'), seed=s.get('seed'),
                 epochs_run=s.get('epochs_run'), best_epoch=s.get('best_epoch'), train_seconds=s.get('train_seconds'),
                 gpu=s.get('gpu'), tf_version=s.get('tf_version'), dice=s.get('dice_per_image'),
                 params_total=s.get('total_params', s.get('params_total')))
        for k in ('grad_audit_init', 'grad_audit_best'):
            g = s.get(k)
            if g:
                a = np.array(g['per_weight'])            # (layers, qubits, 3); index [1,:,2] = final omega (inert)
                inert = a[1, :, 2]; mask = np.ones_like(a, bool); mask[1, :, 2] = False; e = a[mask]
                r.update({f'{k}_max': float(e.max()), f'{k}_median': float(np.median(e)), f'{k}_min': float(e.min()),
                          f'{k}_inert_max': float(inert.max()), f'{k}_n_active_1pct': int((e >= 0.01 * e.max()).sum())})
        h = pd.read_csv(f'{d}/history.csv') if os.path.exists(f'{d}/history.csv') else None
        if h is not None:
            lr = 'learning_rate' if 'learning_rate' in h else 'lr'; b = int(h.val_loss.idxmin())
            low = h.index[h[lr] <= 1e-6]
            r.update(lr_le_1e6_epoch=int(low[0]) + 1 if len(low) else np.nan, lr_at_best=float(h[lr].iloc[b]),
                     train_dice_best=float(h.soft_dice[b]), val_dice_best=float(h.val_soft_dice[b]),
                     val_loss_max_first10=float(h.val_loss[:10].max()))
        rows.append(r)
rr = pd.DataFrame(rows)
rr.to_csv(f'{PROJECT}/results/p2_jit_eval/run_records.csv', index=False)
print(len(rr), 'runs written |', rr.groupby('schedule').size().to_dict())
print(rr.groupby(['schedule', 'model'])[['epochs_run', 'train_seconds']].mean().round(1).to_string())
